# 模块与包

学习目标：组织和导入模块与包，理解搜索路径、导入缓存和程序入口，读取命令行参数、环境变量与包资源。

前置知识：函数与参数、名称绑定和作用域、列表与字典、字符串处理。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

配套脚本：位于 [scripts/08-modules-and-packages/](scripts/08-modules-and-packages/)。

1. [core_tools.py](scripts/08-modules-and-packages/core_tools.py)、[import_notice.py](scripts/08-modules-and-packages/import_notice.py)：演示模块导入、程序入口和导入副作用。
2. [study_reports/](scripts/08-modules-and-packages/study_reports/)：演示常规包、相对导入和包资源。
3. [ns-left/](scripts/08-modules-and-packages/ns-left/)、[ns-right/](scripts/08-modules-and-packages/ns-right/)：演示命名空间包。

## 1 模块与导入

### 1.1 模块、包与标准库

| 名称 | 中文名称／含义 |
| --- | --- |
| module | 模块，组织函数、变量等名称的代码单元 |
| package | 包，用层次化名称组织子模块的特殊模块 |
| standard library | 标准库，随 Python 提供的模块与包 |

本章的模块以 .py 文件为例；模块也可以由扩展库等方式实现，不一定对应一个 Python 源文件。

包也是模块；“包”强调它还可以组织子模块，而不是另一种互不相干的代码对象。

内置函数如 len 可以直接使用，标准库模块如 math 通常需要先导入。

In [1]:
import math

print(math.sqrt(81))  # 9.0：通过模块名访问求平方根的函数。
print(math.__name__)  # math：模块有自己的名称。
print(type(math).__name__)  # module

9.0
math
module


### 1.2 import、from 与 as

下面 module_name 表示模块名，item 表示其中的名称，alias 表示当前代码选用的别名。

| 写法 | 中文名称／含义 |
| --- | --- |
| import module_name | 导入模块，并绑定模块名称 |
| import module_name as alias | 导入模块，并绑定为别名 |
| from module_name import item | 将模块中的一个名称绑定到当前作用域 |
| from module_name import item as alias | 导入指定名称，并使用别名 |

这些语句决定当前作用域得到哪些名称；as 不会复制模块或函数。带点号的子模块导入在包部分展开。

In [2]:
import math as maths
from math import sqrt
from math import sqrt as square_root

print(maths.sqrt(81), sqrt(81), square_root(81))  # 均为 9.0。
print(maths is math)  # True：与上一单元引用同一个模块。
print(square_root is math.sqrt)  # True：别名引用同一个函数。

9.0 9.0 9.0
True
True


### 1.3 查看模块中的名称

dir 用于查看对象提供的名称，返回排序后的字符串列表；不传对象时，查看当前作用域中的名称。

它适合交互探索，不保证列出所有动态属性，也不说明每个名称的用法。进一步阅读说明时使用 help 或官方文档。

In [3]:
import math

visible_names = dir(math)
print([name for name in visible_names if name in {"pi", "sqrt"}])
# ['pi', 'sqrt']：从完整名称列表中挑出本例关心的两项。

print("math" in dir())  # True：当前作用域已经绑定 math。

['pi', 'sqrt']
True


## 2 从文件导入模块

### 2.1 一个可复用的模块

随章的 [core_tools.py](scripts/08-modules-and-packages/core_tools.py) 定义了 passing_scores，用于筛选达到分数线的成绩；threshold 保存默认分数线。文件名去掉 .py 就是这里的导入名。

下方定义 run_python，在示例目录中启动同一环境的独立 Python 进程。-c 表示执行传入的代码文本，-B 避免生成字节码缓存，-X utf8 让中文输入输出使用 UTF-8。

这里只用 pathlib 定位目录、subprocess.run 运行程序：cwd 指定工作目录，check=True 让程序失败时直接报错，capture_output 收集输出。路径与子进程的完整用法在后续专题展开；后文复用这个函数。

In [4]:
import subprocess
import sys
from pathlib import Path

examples_dir = Path("scripts/08-modules-and-packages").resolve()


def run_python(*arguments, env=None):
    """在随章示例目录运行独立 Python 进程，返回其标准输出。"""
    result = subprocess.run(
        [sys.executable, "-B", "-X", "utf8", *arguments],
        cwd=examples_dir,
        env=env,
        check=True,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    return result.stdout.rstrip("\n")


program = "import core_tools\nprint(core_tools.passing_scores([50, 80]))"
print(run_python("-c", program))  # [80]

[80]

### 2.2 from 导入的名称不是实时联动

from 导入把当时取得的对象绑定到当前名称。模块随后重新给同名属性赋值，不会自动重新绑定这个已经导入的名称。

这与普通赋值的引用规则相同；若两边仍引用同一个可变对象，原地修改又是另一种情况。

In [5]:
program = """
import core_tools
from core_tools import threshold

core_tools.threshold = 80
print(threshold, core_tools.threshold)  # 60 80：本地绑定没有随模块属性重绑定。
print(core_tools.passing_scores([70, 90]))  # [90]。
"""
print(run_python("-c", program))
# 第一行：60 80；本地 threshold 仍引用导入时的整数。
# 第二行：[90]；函数读取的是所属模块中的 threshold。

60 80
[90]

### 2.3 首次导入、缓存与副作用

首次加载模块时会执行它的顶层代码，之后通常从 sys.modules 取得同一个模块对象，因此重复 import 不会重新执行全部代码。

[import_notice.py](scripts/08-modules-and-packages/import_notice.py) 故意在顶层打印，用来观察这个过程。普通库模块应把读取输入、启动任务等业务行为放进函数，避免导入就执行。

修改源文件后，普通 import 也不会自动更新缓存；交互学习时可重启内核，重新运行。缓存与重载的完整机制在运行机制专题展开。

In [6]:
program = """
import sys
import import_notice
import import_notice as again

print(again is sys.modules["import_notice"])  # True；此前初始化提示仅出现一次。
"""
print(run_python("-c", program))
# 初始化 import_notice
# True
# 初始化文本只出现一次；别名仍引用缓存中的模块。

初始化 import_notice
True


### 2.4 搜索路径与同名遮蔽

导入会先检查模块缓存，再由导入系统查找；普通文件模块的查找涉及 sys.path，其中包含启动位置、PYTHONPATH 和安装配置等提供的位置。

通常直接运行脚本时，脚本目录在搜索路径前部；使用 -c 或 -m 时，当前工作目录在前部。若本地文件也叫 json.py、random.py 等，可能遮蔽同名标准库模块。

下例读取模块的 \_\_file\_\_，确认加载的是哪个源文件。该属性并非所有模块都有；不要把 sys.path 当作包安装的替代方案。

In [7]:
program = """
from pathlib import Path
import core_tools

print(Path(core_tools.__file__).name)  # core_tools.py。
"""
print(run_python("-c", program))  # core_tools.py

core_tools.py


## 3 包与层次化名称

### 3.1 常规包

常规包（regular package）通常是带有 \_\_init\_\_.py 的目录；首次导入包时会执行这个文件。它可以保持简单，也可以提供少量公开入口。

随章示例按以下职责组织；计算逻辑集中在 metrics.py。

| 文件 | 中文名称／含义 |
| --- | --- |
| [study_reports/\_\_init\_\_.py](scripts/08-modules-and-packages/study_reports/__init__.py) | 包初始化，提供 summarize 入口 |
| [study_reports/metrics.py](scripts/08-modules-and-packages/study_reports/metrics.py) | 计算人数和平均分 |
| [study_reports/\_\_main\_\_.py](scripts/08-modules-and-packages/study_reports/__main__.py) | 包的命令行入口 |
| [study_reports/labels.txt](scripts/08-modules-and-packages/study_reports/labels.txt) | 随包提供的标题文本 |

In [8]:
program = """
import study_reports

print(study_reports.summarize([60, 80, 100]))  # {'count': 3, 'average': 80.0}。
print(study_reports.__name__)  # study_reports。
"""
print(run_python("-c", program))
# {'count': 3, 'average': 80.0}
# study_reports

{'count': 3, 'average': 80.0}
study_reports


### 3.2 绝对导入与相对导入

绝对导入写完整包路径，例如 from study_reports.metrics import summarize。相对导入以点号开头，单点表示当前包，双点表示上一级包。

import study_reports.metrics 绑定名称 study_reports，通过完整路径访问子模块；加上 as metrics 后，则用 metrics 访问该子模块。

本例 \_\_init\_\_.py 中的 from .metrics import summarize，相对于 study_reports 包定位 metrics；这已经在上一单元导入包时执行。

相对导入依赖包上下文，不是按当前工作目录寻找文件；直接运行包内源文件可能缺少这个上下文，下一节用 -m 执行包入口。

In [9]:
program = """
import study_reports.metrics as metrics
from study_reports import summarize

print(metrics.__name__)  # study_reports.metrics。
print(metrics.summarize is summarize)  # True。
"""
print(run_python("-c", program))
# study_reports.metrics
# True：包入口与子模块中的函数是同一个对象。

study_reports.metrics
True

### 3.3 公开名称与通配符导入

\_\_all\_\_ 是控制 from 模块名 import \* 导入哪些名称的字符串列表；没有它时，通常导入当前模块命名空间中不以下划线开头的名称。

对包执行通配符导入，并不会自动发现并加载目录中的所有子模块。\_\_all\_\_ 也不是访问控制，仍可显式导入其他可用名称。

下面仅观察规则；实际代码优先明确写出需要的名称，避免覆盖已有名称。

In [10]:
program = """
from study_reports import *
import study_reports

print(study_reports.__all__)  # ['summarize']。
print("summarize" in globals(), "metrics" in globals())  # True False。
"""
print(run_python("-c", program))
# ['summarize']
# True False：只引入 __all__ 列出的 summarize。

['summarize']
True False


### 3.4 命名空间包

命名空间包（namespace package）没有包根目录的 \_\_init\_\_.py，可以把不同位置中的同名包目录组合到同一个导入名称下。

首次按默认路径查找时，只有未找到同名常规包或模块，才会合并这些目录；本例任一 study_plugins 目录都不应添加 \_\_init\_\_.py。

本例的 ns-left 和 ns-right 都包含 study_plugins，分别提供 text_tools 和 number_tools。仅为演示，在独立进程中把两处目录加入 sys.path；常规项目应通过项目组织和安装来提供导入路径。

In [11]:
program = """
import sys

sys.path.extend(["ns-left", "ns-right"])
from study_plugins import number_tools, text_tools

print(text_tools.words("Python modules"))  # ['Python', 'modules']。
print(number_tools.double(3))  # 6。
"""
print(run_python("-c", program))
# ['Python', 'modules']
# 6：两个子模块来自不同目录中的同一个命名空间包。

['Python', 'modules']
6


## 4 程序入口

### 4.1 直接运行与被导入

作为顶层程序运行时，模块的 \_\_name\_\_ 是 "\_\_main\_\_"；作为普通模块导入时，它是导入名称。

core_tools.py 用 if \_\_name\_\_ == "\_\_main\_\_" 保护 main 调用，因此既能提供可复用函数，也能直接运行一个演示。main 是普通函数名，并没有“定义后自动执行”的特殊规则。

In [12]:
program = "import core_tools\nprint(core_tools.__name__)"
print(run_python("-c", program))
# core_tools：普通导入不会调用该文件中的 main。

print(run_python("core_tools.py"))
# [80, 90]：直接运行文件时，入口中的 main 被调用。

core_tools


[80, 90]


### 4.2 使用 python -m

python -m 后面写可运行模块或包的导入名，不写 .py 后缀。执行包名时，会运行包内的 \_\_main\_\_.py。

在示例目录运行 python -m study_reports 60 80 100，会向包入口传入三项成绩。-m 提供相应的包上下文，因此入口里的 from .metrics import summarize 能正确定位。

带有包内相对导入的入口，应按模块方式运行，避免直接运行 study_reports/\_\_main\_\_.py。

In [13]:
print(run_python("-m", "study_reports", "60", "80", "100"))
# 成绩摘要：3 人，平均分 80.0

# 等价命令在 scripts/08-modules-and-packages 目录执行：
# python -m study_reports 60 80 100

成绩摘要：3 人，平均分 80.0


## 5 常用系统接口与包资源

### 5.1 os：工作目录与环境变量

| 名称 | 中文名称／含义 |
| --- | --- |
| os.getcwd | 取得当前工作目录的字符串 |
| os.environ | 当前进程的环境变量映射 |
| os.getenv | 读取指定环境变量，不存在时使用默认值 |

环境变量的值是字符串，需要数值时再转换。修改当前 Python 进程的环境变量不会改写已经启动它的终端。

下面复制环境映射，只给新启动的进程增加 COURSE_LEVEL；不修改 Notebook 进程的原映射，也不输出其他环境变量。

In [14]:
import os

child_env = os.environ.copy()
child_env["COURSE_LEVEL"] = "2"

program = """
import os

level = int(os.getenv("COURSE_LEVEL", "1"))
print(level + 1)  # 3：环境文本已转成整数。
print(type(os.getcwd()).__name__)  # str；不输出机器相关的绝对路径。
"""
print(run_python("-c", program, env=child_env))
# 3
# str：工作目录由字符串表示。

3
str


### 5.2 sys：命令行参数与标准流

| 名称 | 中文名称／含义 |
| --- | --- |
| sys.argv | 程序接收到的命令行参数列表 |
| sys.stdin | 标准输入流 |
| sys.stdout | 标准输出流 |
| sys.stderr | 标准错误流，常用于错误或诊断信息 |
| sys.executable | 当前 Python 解释器的位置，本章用它运行子进程 |

sys.argv[0] 标识启动入口，其余项是传给程序的字符串；数字文字需要自行转换。标准流的读写与重定向在文件专题展开。

本章的 run_python 已经读取子进程的标准输出，因此下面的文本会显示在 Notebook 中。

In [15]:
program = """
import sys

print(sys.argv[0])  # -c。
print(sys.argv[1:])  # ['Ada', '80']；两项都还是字符串。
print("已收到成绩", file=sys.stdout)  # 最后一行：已收到成绩。
"""
print(run_python("-c", program, "Ada", "80"))
# -c
# ['Ada', '80']
# 已收到成绩

-c
['Ada', '80']
已收到成绩


### 5.3 随包读取资源

包资源是与包一起提供的文本等数据。importlib.resources.files 按包定位资源容器，joinpath 选择其中的资源，read_text 按给定编码读取文本。

这比把资源路径写成相对于当前工作目录的固定字符串更合适；包资源也可能存在于压缩包等形式中，不应假定它总是普通磁盘文件。

In [16]:
program = """
from importlib.resources import files

resource = files("study_reports").joinpath("labels.txt")
print(resource.read_text(encoding="utf-8").strip())  # 成绩摘要。
"""
print(run_python("-c", program))  # 成绩摘要

成绩摘要


## 6 导入时的工程注意

### 6.1 避免循环依赖

循环导入指模块之间的依赖形成环，例如模块 A 导入 B，而 B 又导入 A。若 B 在 A 初始化完成前读取 A 尚未定义的名称，就可能失败；并不是所有导入环都会立即报错。

优先把共同逻辑移到独立模块，使依赖方向清楚；必要时可把仅在调用时需要的导入放进函数，但不要用大量局部导入掩盖模块职责问题。

本例中，包入口和命令行入口都使用 metrics，metrics 只负责计算，不反向依赖这两个入口。

In [17]:
program = """
from study_reports import summarize
from study_reports.metrics import summarize as summarize_directly

print(summarize is summarize_directly)  # True。
print(summarize([]))  # {'count': 0, 'average': None}。
"""
print(run_python("-c", program))
# True：不同入口复用同一处计算逻辑。
# {'count': 0, 'average': None}：计算函数不依赖命令行输入。

True
{'count': 0, 'average': None}


### 6.2 了解 future 语句

from \_\_future\_\_ import ... 是编译器识别的特性声明，不是安装库；普通 import \_\_future\_\_ 则不会启用这些特性。

future 语句须放在模块开头，只能在它之前放文档字符串、注释、空行或其他 future 语句。

下面仅观察 Python 3.12 中的 annotations：value: int 与箭头后的 int 是输入和返回值的类型标注，启用该特性后以字符串保存；标注本身不执行类型检查。类型标注的完整用法在对应专题展开。

In [18]:
program = """
from __future__ import annotations

def double(value: int) -> int:
    return value * 2

print(double.__annotations__)  # {'value': 'int', 'return': 'int'}；本例注解为字符串。
print(double(3))  # 6。
"""
print(run_python("-c", program))
# {'value': 'int', 'return': 'int'}
# 6
# 在独立进程中观察该特性，不改变 Notebook 后续单元的编译方式。

{'value': 'int', 'return': 'int'}
6


## 7 综合应用：从包入口生成摘要

study_reports 将计算、标题资源和命令行入口分开：metrics 计算结果，labels.txt 保存标题，\_\_main\_\_.py 转换参数并输出。

下面用两组输入运行同一个入口。输入约定为整数成绩；未传成绩时得到人数 0 和平均分 None，非整数实参会保留转换错误，不伪造正常结果。

In [19]:
print(run_python("-m", "study_reports", "70", "90"))
# 成绩摘要：2 人，平均分 80.0

print(run_python("-m", "study_reports"))
# 成绩摘要：0 人，平均分 None

# 所有随章示例均未写入数据文件；子进程结束后不保留运行状态。

成绩摘要：2 人，平均分 80.0


成绩摘要：0 人，平均分 None


## 本章小结

（1）import 取得模块并绑定名称；from 导入不会随模块属性重新赋值而自动更新。

（2）包也是模块。常规包使用初始化文件，命名空间包可组合多个位置的内容。

（3）普通导入通常复用缓存；顶层业务代码可能产生导入副作用，循环依赖可能读取尚未完成初始化的名称。

（4）用入口条件区分直接运行与导入；使用 -m 执行包入口并保留包上下文。

（5）用 sys.argv 读取命令行实参，用 os 读取环境变量，用 importlib.resources 定位包资源。

## 练习

（1）从 math 导入 sqrt，命名为 root，计算 49 的平方根。再使用模块别名 math_tools 完成相同计算，并比较两处引用的函数是否相同。

In [20]:
# 在这里完成导入与计算。
# 检查：两种写法均得到 7.0，两个函数引用的身份比较为 True。

（2）先预测下方输出，再运行核对。说明给本地名称重新赋值后，模块属性与函数读取到的值分别是什么。

In [21]:
# 先预测两次打印，再核对本地 threshold 与模块中的 threshold 是否相同。
program = """
import core_tools
from core_tools import threshold

threshold = 90
print(threshold, core_tools.threshold)
print(core_tools.passing_scores([70, 95]))
"""
print(run_python("-c", program))

90 60
[70, 95]


（3）通过 run_python 以模块方式运行 study_reports，传入 60、90、90。检查人数为 3、平均分为 80.0；再改为 50、70，检查人数为 2、平均分为 60.0。

说明入口文件、计算函数与标题资源各自承担什么职责。

In [22]:
# 使用 run_python("-m", 包名, 各项实参) 完成两次运行。
# 注意：命令行实参使用字符串，不直接传入整数对象。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（3.12） | [教程：模块、导入方式与 dir](https://docs.python.org/zh-cn/3.12/tutorial/modules.html)、[包与相对导入](https://docs.python.org/zh-cn/3.12/tutorial/modules.html#packages)；[语言参考：包是特殊模块](https://docs.python.org/zh-cn/3.12/reference/import.html#packages)、[常规包](https://docs.python.org/zh-cn/3.12/reference/import.html#regular-packages)、[命名空间包](https://docs.python.org/zh-cn/3.12/reference/import.html#namespace-packages)、[模块缓存](https://docs.python.org/zh-cn/3.12/reference/import.html#the-module-cache)、[主模块的特殊处理](https://docs.python.org/zh-cn/3.12/reference/import.html#special-considerations-for-main)；[import 语句与公开名称](https://docs.python.org/zh-cn/3.12/reference/simple_stmts.html#the-import-statement)、[future 语句](https://docs.python.org/zh-cn/3.12/reference/simple_stmts.html#future-statements)；[顶层代码与入口](https://docs.python.org/zh-cn/3.12/library/__main__.html#what-is-the-top-level-code-environment)、[包中的主入口](https://docs.python.org/zh-cn/3.12/library/__main__.html#main-py-in-python-packages)；[命令行 -m](https://docs.python.org/zh-cn/3.12/using/cmdline.html#cmdoption-m)、[-c](https://docs.python.org/zh-cn/3.12/using/cmdline.html#cmdoption-c)、[-B](https://docs.python.org/zh-cn/3.12/using/cmdline.html#cmdoption-B)、[-X utf8](https://docs.python.org/zh-cn/3.12/using/cmdline.html#cmdoption-X)；[sys.path](https://docs.python.org/zh-cn/3.12/library/sys.html#sys.path)、[sys.argv](https://docs.python.org/zh-cn/3.12/library/sys.html#sys.argv)、[标准流](https://docs.python.org/zh-cn/3.12/library/sys.html#sys.stdin)、[sys.executable](https://docs.python.org/zh-cn/3.12/library/sys.html#sys.executable)；[os.getcwd](https://docs.python.org/zh-cn/3.12/library/os.html#os.getcwd)、[环境映射](https://docs.python.org/zh-cn/3.12/library/os.html#os.environ)、[os.getenv](https://docs.python.org/zh-cn/3.12/library/os.html#os.getenv)；[包资源 files](https://docs.python.org/zh-cn/3.12/library/importlib.resources.html#importlib.resources.files)；[subprocess.run](https://docs.python.org/zh-cn/3.12/library/subprocess.html#subprocess.run)、[Path.resolve](https://docs.python.org/zh-cn/3.12/library/pathlib.html#pathlib.Path.resolve)、[dir](https://docs.python.org/zh-cn/3.12/library/functions.html#dir)、[math.sqrt](https://docs.python.org/zh-cn/3.12/library/math.html#math.sqrt)；[编程 FAQ：导入位置与循环导入](https://docs.python.org/zh-cn/3.12/faq/programming.html#what-are-the-best-practices-for-using-import-in-a-module)。 |
| Python PEP | [PEP 420：命名空间包的查找与合并条件](https://peps.python.org/pep-0420/#specification)。 |